# 🔤 Old Permic OCR — Progressive Synthetic Dataset Generation

**Layer 2**: Generates synthetic training data through 12 progressive curriculum stages.

| Stage | Focus | Difficulty |
|-------|-------|------------|
| 1 | Clean isolated glyphs | Minimal |
| 2–3 | Paper textures, mild degradation | Low |
| 4–6 | Stone/parchment backgrounds, geometric variation | Medium |
| 7–9 | Multi-material, occlusions, complex scenes | High |
| 10–12 | Historical manuscript simulation | Maximum |

Each stage builds on the previous, ensuring the YOLO model is trained
progressively from simple to hard recognition tasks.

In [ ]:
# ── Cell 02: Runtime & Environment Info ──────────────────────────────────────
import subprocess, sys, platform

print(f'Python : {sys.version.split()[0]}')
print(f'Platform: {platform.system()} {platform.machine()}')

# GPU info
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                        '--format=csv,noheader'], text=True).strip()
    print(f'GPU     : {gpu_info}')
except Exception:
    print('GPU     : Not available (CPU mode)')

# RAM info
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f'RAM     : {ram:.1f} GB')
except ImportError:
    print('RAM     : psutil not installed')

# Disk info
import shutil
disk = shutil.disk_usage('/content' if __import__('os').path.exists('/content') else '.')
print(f'Disk    : {disk.free / 1e9:.1f} GB free of {disk.total / 1e9:.1f} GB')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# ── Cell 03: Install Package ─────────────────────────────────────────────────
import subprocess, sys

print('Installing old-permic-ocr-lab from colab-checkpoints branch...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/Emran025/old-permic-ocr-lab@colab-checkpoints'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ Package installed successfully.')
else:
    print('⚠️  pip install output:')
    print(result.stderr[-2000:])

# Also ensure pyyaml is available
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], capture_output=True)

In [ ]:
# ── Cell 04: GitHub Authentication ───────────────────────────────────────────
import os, getpass, stat, tempfile

def _get_token() -> str:
    # 1. Try Colab Secrets (recommended — never stored in notebook)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t
    except Exception:
        pass
    # 2. Try environment variable
    t = os.environ.get('GITHUB_TOKEN', '')
    if t:
        return t
    # 3. Prompt securely (not echoed)
    return getpass.getpass('Enter GitHub Personal Access Token (hidden): ')

_TOKEN = _get_token()
os.environ['GITHUB_TOKEN'] = _TOKEN

# Write a GIT_ASKPASS helper that echoes the token without storing it
_askpass_script = tempfile.NamedTemporaryFile(
    mode='w', suffix='.sh', delete=False, prefix='/tmp/git_askpass_'
)
_askpass_script.write('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
_askpass_script.close()
os.chmod(_askpass_script.name, stat.S_IRWXU)
os.environ['GIT_ASKPASS'] = _askpass_script.name

print('✅ GitHub token configured (never printed or stored to disk).')

In [ ]:
# ── Cell 05: Clone / Update Repository ───────────────────────────────────────
import os, subprocess

REPO_URL = 'https://github.com/Emran025/old-permic-ocr-lab'
CHECKPOINT_BRANCH = 'colab-checkpoints'
REPO_DIR = '/content/repo'

def _run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, capture_output=True, **kwargs)

_run(['git', 'config', '--global', 'user.email', 'dataset-bot@ocr-lab'])
_run(['git', 'config', '--global', 'user.name', 'Dataset Bot'])

TOKEN = os.environ['GITHUB_TOKEN']
AUTH_URL = REPO_URL.replace('https://', f'https://{TOKEN}@')

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Updating existing clone...')
    _run(['git', '-C', REPO_DIR, 'fetch', '--all'])
    _run(['git', '-C', REPO_DIR, 'checkout', CHECKPOINT_BRANCH])
    _run(['git', '-C', REPO_DIR, 'pull', 'origin', CHECKPOINT_BRANCH])
else:
    print('Cloning repository...')
    subprocess.run(
        ['git', 'clone', '--branch', CHECKPOINT_BRANCH, AUTH_URL, REPO_DIR],
        check=True, capture_output=True
    )
del AUTH_URL, TOKEN  # Remove from local scope immediately

# Verify structure
GLYPH_ROOT = f'{REPO_DIR}/font/svg'
if os.path.isdir(GLYPH_ROOT):
    families = [d for d in os.listdir(GLYPH_ROOT) if os.path.isdir(f'{GLYPH_ROOT}/{d}')]
    print(f'✅ Repository ready. Font families: {families}')
else:
    print(f'⚠️  font/svg not found at {GLYPH_ROOT}. Check repo structure.')

In [ ]:
# ── Cell 06: Smoke Test Imports ───────────────────────────────────────────────
from historical_glyph_studio import GlyphStudio
from historical_glyph_curriculum import STAGES, get_stage, GenerationPlan
from historical_glyph_curriculum.github_sync.git_ops import GitManager
from historical_glyph_curriculum.resources.detection import detect_resources

print(f'✅ historical_glyph_studio imported')
print(f'✅ historical_glyph_curriculum imported — {len(STAGES)} stages defined')

for i, stage in enumerate(STAGES[:3], 1):
    print(f'   Stage {i}: {stage.name} ({len(stage.concepts)} concepts)')

## ⚙️ Configuration

Edit these values before running the generation cells.

- `GENERATION_MODE`: `'dev'` (50 samples, fast test) | `'medium'` (500) | `'full'` (stage default)
- `GLYPH_ROOT`: Path to `font/svg` directory (auto-set from repo clone above)
- `OUTPUT_ROOT`: Where to save generated datasets

In [ ]:
# ── Cell 08: Configuration ────────────────────────────────────────────────────
import os

# ── Primary settings ──────────────────────────────────────────────────────────
GENERATION_MODE = 'medium'      # 'dev' | 'medium' | 'full'
GLYPH_ROOT      = '/content/repo/font/svg'
OUTPUT_ROOT     = '/content/datasets'
GITHUB_REPO     = 'https://github.com/Emran025/old-permic-ocr-lab'
CHECKPOINT_BRANCH = 'colab-checkpoints'
SESSION_FILE    = '/content/generation_session.json'

# Samples per stage per mode (None = use stage default)
SAMPLES_BY_MODE = {
    'dev':    50,
    'medium': 500,
    'full':   None,
}
SAMPLES_OVERRIDE = SAMPLES_BY_MODE[GENERATION_MODE]

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f'Mode            : {GENERATION_MODE}')
print(f'Samples/stage   : {SAMPLES_OVERRIDE or "stage default"}')
print(f'Glyph root      : {GLYPH_ROOT}')
print(f'Output root     : {OUTPUT_ROOT}')

In [ ]:
# ── Cell 09: Resource Detection & Auto-tune ───────────────────────────────────
from historical_glyph_curriculum.resources.detection import detect_resources, auto_tune

resources = detect_resources()
tuned = auto_tune(resources, GENERATION_MODE)

print(f'GPU VRAM  : {resources.gpu_vram_gb:.1f} GB')
print(f'CPU cores : {resources.cpu_count}')
print(f'RAM       : {resources.ram_gb:.1f} GB')
print()
print(f'Auto-tuned workers  : {tuned.workers}')
print(f'Auto-tuned batch_sz : {tuned.batch_size}')

In [ ]:
# ── Cell 10: Engine Setup ─────────────────────────────────────────────────────
import os
from historical_glyph_studio import GlyphStudio
from historical_glyph_curriculum.github_sync.git_ops import GitManager

# Create the rendering studio
studio = GlyphStudio(glyph_root=GLYPH_ROOT)
print(f'✅ GlyphStudio ready — {studio.repository.count()} glyphs indexed')

# Create git manager for pushing datasets
git_manager = GitManager(
    repo_url=GITHUB_REPO,
    checkpoint_branch=CHECKPOINT_BRANCH,
    work_dir='/content/repo',
)
git_manager.setup(token=os.environ.get('GITHUB_TOKEN', ''))
print(f'✅ GitManager ready — pushing to branch: {CHECKPOINT_BRANCH}')

In [ ]:
# ── Cell 11: Approval Gate Helper ────────────────────────────────────────────
import time

def approval_gate(title: str, details: str = '') -> bool:
    """Interactive approval gate — ipywidgets when available, input() fallback."""
    print(f"\n{'━'*60}")
    print(f'  {title}')
    print(f"{'━'*60}")
    if details:
        print(details)

    try:
        import ipywidgets as w
        from IPython.display import display
        decision = [None]
        btn_ok  = w.Button(description='✅ Approve & Continue', button_style='success',
                           layout=w.Layout(width='220px'))
        btn_skip = w.Button(description='⏭ Skip Stage', button_style='warning',
                            layout=w.Layout(width='160px'))
        btn_stop = w.Button(description='🛑 Stop', button_style='danger',
                            layout=w.Layout(width='120px'))
        out = w.Output()
        def _approve(_): decision[0] = True;  out.clear_output(); print('Approved ✅')
        def _skip(_):    decision[0] = 'skip'; out.clear_output(); print('Skipped ⏭')
        def _stop(_):    decision[0] = False;  out.clear_output(); print('Stopped 🛑')
        btn_ok.on_click(_approve)
        btn_skip.on_click(_skip)
        btn_stop.on_click(_stop)
        display(w.HBox([btn_ok, btn_skip, btn_stop]), out)
        while decision[0] is None:
            time.sleep(0.3)
        return decision[0]
    except Exception:
        resp = input('Approve stage? [y/skip/n]: ').strip().lower()
        if resp in ('y', 'yes', 'approve', ''):
            return True
        if resp == 'skip':
            return 'skip'
        return False

print('✅ Approval gate helper ready.')

In [ ]:
# ── Cell 12: Stage Runner Function ───────────────────────────────────────────
import os, json, time
from pathlib import Path
from historical_glyph_curriculum import get_stage
from historical_glyph_curriculum.parallel.executor import CurriculumExecutor
from historical_glyph_curriculum.validation.dataset import DatasetValidator
from historical_glyph_curriculum.metadata.manifest import save_stage_manifest
from historical_glyph_curriculum.metadata.report import print_stage_summary
from historical_glyph_curriculum.preview.grid import display_grid_in_colab, select_preview_samples

def run_stage(
    stage_id: int,
    studio,
    git_manager,
    mode: str = 'medium',
    session_file: str = SESSION_FILE,
) -> bool:
    """
    Generate, validate, preview, and commit one curriculum stage.
    Returns True if stage was approved and committed.
    """
    stage_out = Path(OUTPUT_ROOT) / f'stage_{stage_id:02d}'
    manifest_path = stage_out / 'stage_manifest.json'

    # Check if already done
    if manifest_path.exists():
        print(f'\n✅ Stage {stage_id:02d} already completed (manifest exists). Skipping.')
        return True

    stage_def = get_stage(stage_id)
    print(f'\n{"═"*60}')
    print(f'  📚 Stage {stage_id:02d}: {stage_def.name}')
    print(f'  Concepts: {len(stage_def.concepts)} | Mode: {mode}')
    print(f'{"═"*60}')
    stage_out.mkdir(parents=True, exist_ok=True)

    # Generate dataset
    executor = CurriculumExecutor(studio=studio, output_root=str(stage_out))
    samples_per_concept = SAMPLES_BY_MODE.get(mode)
    t0 = time.time()
    manifest = executor.generate_stage(
        stage_id=stage_id,
        samples_override=samples_per_concept,
    )
    elapsed = time.time() - t0
    print(f'  Generated {manifest.total_samples} samples in {elapsed:.1f}s')

    # Preview grid
    try:
        samples = select_preview_samples(manifest, n=16)
        display_grid_in_colab(samples, title=f'Stage {stage_id:02d} Preview')
    except Exception as e:
        print(f'  Preview skipped: {e}')

    # Validate
    validator = DatasetValidator()
    report = validator.validate(str(stage_out))
    print_stage_summary(report)

    if not report.is_valid:
        print(f'  ❌ Validation failed: {report.errors[:3]}')
        return False

    # Approval gate
    details = (
        f'  Samples    : {manifest.total_samples}\n'
        f'  Validation : {"PASS" if report.is_valid else "FAIL"}\n'
        f'  Time       : {elapsed:.1f}s'
    )
    decision = approval_gate(f'Stage {stage_id:02d} — {stage_def.name}', details)

    if decision is False:
        print('  Generation stopped by user.')
        return False
    if decision == 'skip':
        print('  Stage skipped by user.')
        return True

    # Save manifest and commit
    save_stage_manifest(manifest, str(manifest_path))
    commit_files = [str(manifest_path)]
    git_manager.push_stage(
        stage_id=stage_id,
        files=commit_files,
        output_dir=str(stage_out),
    )
    print(f'  ✅ Stage {stage_id:02d} committed to {CHECKPOINT_BRANCH}')
    return True

print('✅ run_stage() function ready.')

In [ ]:
# ── Stages 01–04: Foundation ──────────────────────────────────────────────────
for stage_id in range(1, 5):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 1–4 complete!')

In [ ]:
# ── Stages 05–08: Intermediate ────────────────────────────────────────────────
for stage_id in range(5, 9):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 5–8 complete!')

In [ ]:
# ── Stages 09–12: Advanced Historical ────────────────────────────────────────
for stage_id in range(9, 13):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 9–12 complete!')

In [ ]:
# ── Cell 16: Master Manifest + Final Report ───────────────────────────────────
import json, os
from pathlib import Path
from historical_glyph_curriculum.metadata.manifest import build_master_manifest, save_stage_manifest

print('Building master curriculum manifest...')

stage_manifests = []
summary_rows = []

for stage_id in range(1, 13):
    stage_dir = Path(OUTPUT_ROOT) / f'stage_{stage_id:02d}'
    manifest_path = stage_dir / 'stage_manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            data = json.load(f)
        stage_manifests.append(data)
        total = data.get('total_samples', '?')
        summary_rows.append(f'  ✅ Stage {stage_id:02d}: {total} samples')
    else:
        summary_rows.append(f'  ⏳ Stage {stage_id:02d}: not generated')

master = build_master_manifest(stage_manifests)
master_path = Path(OUTPUT_ROOT) / 'curriculum_manifest.json'
master_path.write_text(json.dumps(master, indent=2, ensure_ascii=False))

print('\n═══════════════════════════════════════════════════════')
print('  OLD PERMIC OCR — DATASET GENERATION SUMMARY')
print('═══════════════════════════════════════════════════════')
for row in summary_rows:
    print(row)
print(f'\n  Master manifest: {master_path}')
total_completed = sum(1 for r in summary_rows if '✅' in r)
print(f'  Stages completed: {total_completed} / 12')
print('═══════════════════════════════════════════════════════')

if total_completed == 12:
    print('\n🎉 All 12 stages generated! Ready for adaptive_training.ipynb')
else:
    print(f'\n⚠️  {12 - total_completed} stage(s) still needed. Re-run generation cells.')

# Push master manifest
try:
    git_manager.push_stage(
        stage_id=0,
        files=[str(master_path)],
        output_dir=OUTPUT_ROOT,
    )
    print('✅ Master manifest pushed to repository.')
except Exception as e:
    print(f'⚠️  Could not push master manifest: {e}')